In [5]:
import sqlite3
import pandas as pd
import re
import torch

from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import AutoTokenizer

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

from sklearn.metrics import accuracy_score

from sklearn.metrics import classification_report

from sklearn.metrics import f1_score

import json

import torch





#pull the data (SQL)
conn = sqlite3.connect("products.db")
df = pd.read_sql("""
    SELECT
        p.uuid,
        p.title,
        p.description,
        t.level_1,
        t.level_2,
        t.taxonomy_path
    FROM products p
    JOIN taxonomy t ON p.taxonomy_id = t.taxonomy_id
    WHERE p.title != ''
      AND t.level_1 IS NOT NULL
""", conn)
conn.close()

print(f"{len(df)} products across {df['level_1'].nunique()} categories")
print(df["level_1"].value_counts())




11771 products across 11 categories
level_1
Tools                                      3813
Hardware Accessories                       1878
Plumbing                                   1498
Building Materials                         1188
Power & Electrical Supplies                1029
Building Consumables                        965
Tool Accessories                            560
Heating, Ventilation & Air Conditioning     280
Fencing & Barriers                          210
Locks & Keys                                210
Hardware Pumps                              140
Name: count, dtype: int64


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [7]:

#step2 preprocessing
# print out numbers of missing empty titels, numbers of null level_1s, numbers of cases where desctiption == title (unnecessary duplication)
print("Empty titles: ", (df["title"] == "").sum())
print("Null level_1: ", df["level_1"].isna().sum())
print("Description == title: ", (df["title"] == df["description"]).sum())

#clean the text

def clean_text(text):
    if pd.isna(text) or text == "":
        return ""
    #remove trailing site names after | or -
    text = re.sub(r'\s*[\|–—]\s*[^|–—]+$', '', text)
    #remove trailing ellipsis
    text = re.sub(r'\.{2,}$', '', text)
    #normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["text"] = df["title"].apply(clean_text)

#spot check
print(df[["title", "text"]].sample(5, random_state=42).to_string())

#check class distribution
print(df["level_1"].value_counts())



Empty titles:  0
Null level_1:  0
Description == title:  11771
                                                                        title                                                                text
3128   ELKAY, On-Wall, Refrigerated, Drinking Fountain with Bottle Filler ...  ELKAY, On-Wall, Refrigerated, Drinking Fountain with Bottle Filler
10010                    Essentials of Table Saw Safety | Popular Woodworking                                      Essentials of Table Saw Safety
200                  Boost HVAC Efficiency: Best Vent Registers & HVAC Covers            Boost HVAC Efficiency: Best Vent Registers & HVAC Covers
9479                                    Waterproof Coating Sealant – Tecnoant                                          Waterproof Coating Sealant
8235                           20V 2.0Ah Lithium-Ion Battery | PowerSmart USA                                       20V 2.0Ah Lithium-Ion Battery
level_1
Tools                                      3813
Hardw

In [8]:
#step3 train/test split
train_df, test_df = train_test_split(
    df, test_size = 0.2, random_state = 42,
stratify=df["level_1"]
)

print(f"Train: {len(train_df)} | Test: {len(test_df)}\n")
print(f"-----------------------------------------------\n")
print(f"Train class distribution:\n{train_df['level_1'].value_counts()}")
print(f"-----------------------------------------------\n")

print(f"-----------------------------------------------")
print(f"Test class distribution:\n{test_df['level_1'].value_counts()}")
print(f"-----------------------------------------------")



Train: 9416 | Test: 2355

-----------------------------------------------

Train class distribution:
level_1
Tools                                      3050
Hardware Accessories                       1502
Plumbing                                   1198
Building Materials                          951
Power & Electrical Supplies                 823
Building Consumables                        772
Tool Accessories                            448
Heating, Ventilation & Air Conditioning     224
Fencing & Barriers                          168
Locks & Keys                                168
Hardware Pumps                              112
Name: count, dtype: int64
-----------------------------------------------

-----------------------------------------------
Test class distribution:
level_1
Tools                                      763
Hardware Accessories                       376
Plumbing                                   300
Building Materials                         237
Power & Electrical 

In [9]:
#step4 fine-tne BERT
#prepare the HuggingFace dataset

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#enconde labels as integers
label_names = sorted(df["level_1"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

train_df["label"] = train_df["level_1"].map(label2id)
test_df["label"] = test_df["level_1"].map(label2id)



#convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

#tokenize
def tokenize(batch):
    return tokenizer(batch["text"], truncation = True, padding="max_length", max_length = 128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map:   0%|          | 0/9416 [00:00<?, ? examples/s]

Map:   0%|          | 0/2355 [00:00<?, ? examples/s]

In [10]:

#------------------------
#train
#------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="./taxonomy_model",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\leehs\anaconda3\Lib\site-pac

Epoch,Training Loss,Validation Loss,Accuracy
1,0.137974,0.106122,0.976645


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\leehs\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [12]:


preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

print(classification_report(
    y_true, y_pred,
    target_names=label_names,
))

#macros vs weighted F1

print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Wighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")

C:\Users\leehs\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.9732
                                         precision    recall  f1-score   support

                   Building Consumables       0.93      0.99      0.96       193
                     Building Materials       0.99      0.99      0.99       237
                     Fencing & Barriers       0.93      1.00      0.97        42
                   Hardware Accessories       0.97      0.98      0.98       376
                         Hardware Pumps       1.00      0.89      0.94        28
Heating, Ventilation & Air Conditioning       1.00      0.98      0.99        56
                           Locks & Keys       0.98      0.98      0.98        42
                               Plumbing       0.97      0.97      0.97       300
            Power & Electrical Supplies       0.99      0.98      0.99       206
                       Tool Accessories       0.95      0.92      0.94       112
                                  Tools       0.98      0.97      0.97       763

         

In [13]:

trainer.save_model("./taxonomy_model/best")
tokenizer.save_pretrained("./taxonomy_model/best")

with open("./taxonomy_model/best/label_mapping.json","w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent = 2)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]